# 06 · Seed 42 — Why do CPU and CUDA INT8 predictions disagree?

**Separate diagnostic notebook — does not modify original models, inference scripts, or HANS result files.**

## Research controls

1. Recreate the **original full-dynamic-W8A8 INT8** inference on **CPU**, with the same batch order and dynamic padding as the original `quantization.ipynb`.
2. Compare to the original `results/hans/int8_seed42_predictions.csv`.
3. Reuse the already-saved **CUDA INT8 eager** predictions for the CPU-vs-CUDA comparison.
4. Create just one new inference variant, **CUDA INT8 eager with classifier retained in FP32**, to isolate the classifier effect from compilation.
5. Compare that new variant against the previously saved **compiled INT8 with FP32 head** to isolate execution-mode effects.

**Start with 10 batches (320 examples)** as a smoke test. Then set `MAX_BATCHES = None` to run the full HANS set. Always keep the same original HANS row order and batch size of 32. Shortening *within* the same sequence of batches preserves the original batch composition; randomly sampling and re-batching examples would not.

### Interpretations

- Original CPU ≈ rerun CPU, but CPU ≠ CUDA: evidence for backend-dependent quantized numerics.
- CUDA full INT8 ≠ CUDA eager FP32-head: classifier precision contributes.
- CUDA eager FP32-head ≠ CUDA compiled FP32-head: compilation/execution contributes.
- Differences may also interact; report observed disagreements rather than assigning causation beyond the controlled contrasts.

## 1 — User settings (normally only change `MAX_BATCHES`)

In [ ]:
SEED = 42
BATCH_SIZE = 32
MAX_LENGTH = 128

# Initial smoke test: exactly the first 10 original HANS batches.
# Change to None for the whole 30,000-example set.
MAX_BATCHES = 10

RUN_CPU_REPRODUCTION = True
RUN_CUDA_EAGER_FP32_HEAD = True

from pathlib import Path

def find_repo_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'models' / 'fp32' / f'seed{SEED}' / 'final').is_dir()            and (candidate / 'results' / 'hans').is_dir():
            return candidate
    raise FileNotFoundError(
        'Cannot find repository root. Run notebook from inside the project or its notebooks directory.'
    )

ROOT = find_repo_root()
MODEL_PATH = ROOT / 'models' / 'fp32' / f'seed{SEED}' / 'final'
HANS_FP32_FILE = ROOT / 'results' / 'hans' / f'fp32_seed{SEED}_predictions.csv'
ORIGINAL_INT8_FILE = ROOT / 'results' / 'hans' / f'int8_seed{SEED}_predictions.csv'
RUNTIME_DIR = ROOT / 'results' / 'runtime_analysis' / f'seed{SEED}'
DIAG_DIR = RUNTIME_DIR / 'diagnostics'
DIAG_DIR.mkdir(parents=True, exist_ok=True)
print('Repository:', ROOT)
print('Model:', MODEL_PATH)
print('Diagnostic output:', DIAG_DIR)

## 2 — Imports and verify reference files

In [ ]:
import copy
import gc
import json
import time

import numpy as np
import pandas as pd
import torch
import torchao

from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
)
from torchao.quantization import quantize_, Int8DynamicActivationInt8WeightConfig

for path in [HANS_FP32_FILE, ORIGINAL_INT8_FILE]:
    if not path.exists():
        raise FileNotFoundError(f'Missing required original prediction CSV: {path}')

print('PyTorch:', torch.__version__, 'TorchAO:', torchao.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Original HANS predictions:', ORIGINAL_INT8_FILE)

## 3 — Build exactly the original HANS batch sequence

In [ ]:
hans = pd.read_csv(HANS_FP32_FILE)
required = {'pairID', 'premise', 'hypothesis'}
if not required.issubset(hans.columns):
    raise ValueError('HANS FP32 predictions CSV is missing: ' + str(required - set(hans.columns)))
if len(hans) != 30000 or hans.pairID.duplicated().any():
    raise ValueError('Expected 30,000 distinct HANS pairIDs')

selected_count = len(hans) if MAX_BATCHES is None else min(len(hans), MAX_BATCHES * BATCH_SIZE)
hans_selected = hans.iloc[:selected_count].reset_index(drop=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
encoded = tokenizer(
    hans_selected.premise.tolist(), hans_selected.hypothesis.tolist(),
    truncation=True, max_length=MAX_LENGTH, padding=False,
)

class TokenizedRows:
    def __init__(self, encoding):
        self.encoding = encoding
    def __len__(self):
        return len(self.encoding['input_ids'])
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encoding.items()}

hans_loader = DataLoader(
    TokenizedRows(encoded), batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=DataCollatorWithPadding(tokenizer, return_tensors='pt'),
    num_workers=0,
)
print('Rows:', len(hans_selected), 'Batches:', len(hans_loader))
print('First IDs:', hans_selected.pairID.head().tolist())

## 4 — Prediction helper (no benchmarking; no tokenization inside timing)

In [ ]:
def predict(model, device, label):
    model = model.to(device).eval()
    all_probs = []
    start = time.perf_counter()
    with torch.no_grad():
        for i, batch in enumerate(hans_loader):
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(**batch).logits
            all_probs.append(torch.softmax(logits.float(), dim=-1).cpu())
            if (i + 1) % 100 == 0:
                print(label, 'batches processed:', i + 1, '/', len(hans_loader))
    probs = torch.cat(all_probs, dim=0).numpy()
    mnli_pred = probs.argmax(axis=1)
    result = pd.DataFrame({
        'pairID': hans_selected.pairID.astype(str).tolist(),
        'mnli_prediction_id': mnli_pred,
        'hans_prediction': np.where(mnli_pred == 0, 'entailment', 'non-entailment'),
        'prob_entailment': probs[:, 0],
        'prob_neutral': probs[:, 1],
        'prob_contradiction': probs[:, 2],
    })
    print(label, 'completed:', len(result), 'examples in', round(time.perf_counter()-start, 1), 's')
    return result

## 5 — Reproduce the original CPU INT8 output

In [ ]:
cpu_reproduction = None
if RUN_CPU_REPRODUCTION:
    # Just like quantization.ipynb: load FP32 on CPU, quantize on CPU,
    # leave CPU model and input batches on CPU for inference.
    cpu_model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).eval()
    quantize_(cpu_model, Int8DynamicActivationInt8WeightConfig())
    cpu_reproduction = predict(cpu_model, torch.device('cpu'), 'original INT8 on CPU')
    cpu_reproduction.to_csv(DIAG_DIR / 'cpu_original_int8_reproduction.csv', index=False)
    del cpu_model
    gc.collect()
else:
    print('Skipped CPU reproduction')

## 6 — Run only the missing control: INT8 eager CUDA with an FP32 classifier

In [ ]:
cuda_fp32_head = None
if RUN_CUDA_EAGER_FP32_HEAD:
    if not torch.cuda.is_available():
        print('CUDA unavailable: skipping the GPU classifier control')
    else:
        head_model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).eval()
        def without_classifier(module, name):
            return isinstance(module, torch.nn.Linear) and name.split('.')[-1] != 'classifier'
        quantize_(
            head_model, Int8DynamicActivationInt8WeightConfig(),
            filter_fn=without_classifier,
        )
        assert isinstance(head_model.classifier, torch.nn.Linear)
        assert head_model.classifier.out_features == 3
        assert head_model.classifier.weight.dtype == torch.float32
        cuda_fp32_head = predict(head_model, torch.device('cuda'), 'INT8 eager CUDA / FP32 head')
        cuda_fp32_head.to_csv(DIAG_DIR / 'cuda_int8_eager_fp32_head.csv', index=False)
        del head_model
        torch.cuda.empty_cache()
        gc.collect()
else:
    print('Skipped CUDA FP32-head control')

## 7 — Compare all available saved and newly generated predictions by `pairID`

In [ ]:
def read_existing(path, name):
    if not path.is_file():
        print('Missing optional variant:', name, path)
        return None
    frame = pd.read_csv(path)
    return frame[frame.pairID.astype(str).isin(hans_selected.pairID.astype(str))].copy()

variants = {
    'original_int8_cpu_reference': read_existing(ORIGINAL_INT8_FILE, 'original INT8 CPU'),
    'int8_eager_cuda_saved': read_existing(
        RUNTIME_DIR / 'int8_eager' / 'predictions.csv', 'int8_eager CUDA'),
    'int8_compiled_fp32_head_saved': read_existing(
        RUNTIME_DIR / 'int8_compiled_fp32_head' / 'predictions.csv', 'compiled INT8 FP32 head'),
    'int8_cpu_reproduced': cpu_reproduction,
    'int8_eager_cuda_fp32_head_new': cuda_fp32_head,
}

probs = ['prob_entailment', 'prob_neutral', 'prob_contradiction']
def compare(first_name, second_name):
    a, b = variants[first_name], variants[second_name]
    if a is None or b is None:
        return None
    a = a.set_index('pairID').sort_index()
    b = b.set_index('pairID').sort_index()
    ids = a.index.intersection(b.index)
    if len(ids) != len(hans_selected):
        raise ValueError(f'Comparison {first_name} vs {second_name}: only {len(ids)} matched IDs')
    a, b = a.loc[ids], b.loc[ids]
    p1, p2 = a[probs].to_numpy(float), b[probs].to_numpy(float)
    mnli_changed = a.mnli_prediction_id.to_numpy() != b.mnli_prediction_id.to_numpy()
    hans_changed = a.hans_prediction.to_numpy() != b.hans_prediction.to_numpy()
    margins = np.sort(p1, axis=1)[:, -1] - np.sort(p1, axis=1)[:, -2]
    return {
        'comparison': first_name + ' vs ' + second_name,
        'n_examples': len(ids),
        'mnli_3way_changes': int(mnli_changed.sum()),
        'hans_binary_changes': int(hans_changed.sum()),
        'hans_binary_change_pct': float(100 * hans_changed.mean()),
        'probability_MAE': float(np.abs(p1-p2).mean()),
        'probability_p99_abs_difference': float(np.quantile(np.abs(p1-p2).ravel(), .99)),
        'flipped_fraction_reference_margin_below_0_1': (
            float((margins[mnli_changed] < .1).mean()) if mnli_changed.any() else np.nan
        ),
    }

pairs = [
    ('original_int8_cpu_reference', 'int8_cpu_reproduced'),
    ('original_int8_cpu_reference', 'int8_eager_cuda_saved'),
    ('int8_eager_cuda_saved', 'int8_eager_cuda_fp32_head_new'),
    ('int8_eager_cuda_fp32_head_new', 'int8_compiled_fp32_head_saved'),
    ('original_int8_cpu_reference', 'int8_compiled_fp32_head_saved'),
]
rows = [compare(a, b) for a, b in pairs]
results = pd.DataFrame([r for r in rows if r is not None])
display(results)
results.to_csv(DIAG_DIR / 'diagnostic_comparisons.csv', index=False)
print('Saved:', DIAG_DIR / 'diagnostic_comparisons.csv')

## 8 — Interpreting the test and next steps

**If the first comparison shows almost identical predictions and probabilities:** We have reproduced the original CPU model. If it still differs from CUDA in the second comparison, this supports a quantized CPU-vs-CUDA execution effect rather than a wrong checkpoint or wrong labels.

**If CUDA full-INT8 differs from CUDA FP32-head eager:** Classifier precision contributes to the change. This isolates head precision while holding backend and execution style constant.

**If CUDA FP32-head eager differs from compiled FP32-head:** Compilation (including kernel selection, operation ordering, and backend numerical behavior) contributes. Do not attribute this to classifier precision, which is matched.

**If CPU reproduction does not match the original CPU predictions:** Before blaming CUDA, check TorchAO/PyTorch versions, exact checkpoint, preprocessing, batching, and whether the old notebook really ran on CPU.

Important: This is a **prediction-consistency** study. Do not use its elapsed times as comparative benchmarks because we are running variants in the same notebook process, not isolated workers.

### Deadline decision

For the November 12 presentation, retain the original FP32–INT8 HANS results as the core study. Present the compiled FP32-head result separately as a promising seed-42 deployment experiment, clearly labeled as a different quantization configuration. Do not assert prediction equivalence when disagreements remain.